# TrustiPay Backend API Integration Notebook

This notebook connects to the FastAPI backend and validates the full reporting pipeline end-to-end.

Expected backend base URL: `http://127.0.0.1:8000`

In [ ]:
# %pip install -q requests pandas

In [ ]:
import os
import requests
import pandas as pd

BASE_URL = os.getenv('TRUSTIPAY_API_BASE_URL', 'http://127.0.0.1:8000')
TIMEOUT = 20

def api_get(path: str, params: dict | None = None):
    url = f'{BASE_URL}{path}'
    response = requests.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()

print('Backend:', BASE_URL)

In [ ]:
# Health check
health = api_get('/health')
health

In [ ]:
# Get available users
users_payload = api_get('/v1/users')
users = users_payload.get('users', [])
print('Users:', len(users))
display(pd.DataFrame(users).head(10))

if not users:
    raise RuntimeError('No users found. Ingest or seed data first.')

USER_REF = users[0]['user_ref'] if isinstance(users[0], dict) else users[0]
print('Selected user:', USER_REF)

In [ ]:
# Optional date window (set to None to use defaults)
FROM = None  # e.g. '2025-01-01'
TO = None    # e.g. '2025-12-31'

params = {}
if FROM:
    params['from'] = FROM
if TO:
    params['to'] = TO

summary = api_get(f'/v1/users/{USER_REF}/reports/summary', params={**params, 'groupBy': 'month'})
categories = api_get(f'/v1/users/{USER_REF}/reports/categories', params=params)
anomalies = api_get(f'/v1/users/{USER_REF}/reports/anomalies', params=params)
features = api_get(f'/v1/users/{USER_REF}/reports/features', params=params)
fhs = api_get(f'/v1/users/{USER_REF}/reports/fhs', params=params)
profile = api_get(f'/v1/users/{USER_REF}/reports/behavior-profile', params=params)
recommendations = api_get(f'/v1/users/{USER_REF}/reports/recommendations', params=params)

print('Fetched all report endpoints successfully.')

In [ ]:
# Quick dashboard-style summary
overview = {
    'user_ref': USER_REF,
    'income_total': summary.get('income_total'),
    'expense_total': summary.get('expense_total'),
    'net_total': summary.get('net_total'),
    'anomaly_count': anomalies.get('anomaly_count'),
    'fhs_score': fhs.get('score'),
    'fhs_interpretation': fhs.get('interpretation'),
    'behavior_profile': profile.get('profile'),
}
pd.DataFrame([overview])

In [ ]:
display(pd.DataFrame(summary.get('series', [])))
display(pd.DataFrame(categories.get('items', [])))
display(pd.DataFrame(anomalies.get('items', [])))
display(pd.DataFrame(recommendations.get('items', [])))

## Optional: ingest sample data before running reports

If `/v1/users` is empty, run the API and seed script from terminal:

```bash
uvicorn app.main:app --reload
python scripts/seed_demo.py
```